# Dengue: Model Sederhana, Vektor–Inang, dan Keteridentifikasian

**ID proyek:** `O005-LEGA-V101-PRJ07`  
**Status:** titik awal pedagogis yang ditulis secara independen.

Notebook ini menggunakan data sintetis/terbuka saja. Notebook ini **bukan** kode atau data dari makalah yang dikutip dalam bab sumber dan **bukan** klaim reproduksi hasil penelitian mana pun.


## Pertanyaan pemodelan

Dapatkah kurva prevalensi manusia saja membedakan laju penularan manusia-ke-vektor dari vektor-ke-manusia?

Tujuan kerja: tetapkan sistem, jalankan eksperimen deterministik, periksa invarian, visualisasikan perilaku, lalu kritik kecukupan model.


In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

SEED = 2026082207
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=6, suppress=True)


## Struktur dan asumsi

Data sintetis berasal dari model vektor–inang Euler dengan populasi ternormalisasi; model SIR sederhana dan kisi pasangan laju dibandingkan pada pengamatan yang sama.

Semua skala dan parameter di notebook ini bersifat ilustratif. Ubah satu asumsi pada satu waktu dan catat dampaknya pada keluaran serta invarian.


In [ ]:
dt, steps = 0.25, 360
times = dt * np.arange(steps + 1)

def simulate_vector_host(beta_hv, beta_vh):
    y = np.empty((steps + 1, 5), dtype=float)
    y[0] = [0.995, 0.005, 0.0, 0.995, 0.005]
    gamma_h, mu_v = 0.12, 0.08
    for k in range(steps):
        Sh, Ih, Rh, Sv, Iv = y[k]
        flux_h = beta_vh * Sh * Iv
        flux_v = beta_hv * Sv * Ih
        dy = np.array([-flux_h, flux_h - gamma_h * Ih, gamma_h * Ih, mu_v - flux_v - mu_v * Sv, flux_v - mu_v * Iv])
        y[k + 1] = y[k] + dt * dy
    return y

def simulate_simple(beta):
    y = np.empty((steps + 1, 3), dtype=float)
    y[0] = [0.995, 0.005, 0.0]
    gamma_h = 0.12
    for k in range(steps):
        S, I, R = y[k]
        flux = beta * S * I
        y[k + 1] = y[k] + dt * np.array([-flux, flux - gamma_h * I, gamma_h * I])
    return y

truth = simulate_vector_host(0.78, 0.52)
obs_idx = np.arange(0, steps + 1, 12)
observed = np.clip(truth[obs_idx, 1] + rng.normal(0.0, 0.0006, obs_idx.size), 0.0, None)

simple_betas = np.linspace(0.08, 0.80, 73)
simple_errors = np.array([np.mean((simulate_simple(b)[obs_idx, 1] - observed) ** 2) for b in simple_betas])
best_simple = simulate_simple(float(simple_betas[np.argmin(simple_errors)]))

grid = np.linspace(0.30, 1.00, 19)
surface = np.empty((grid.size, grid.size))
for i, beta_hv in enumerate(grid):
    for j, beta_vh in enumerate(grid):
        surface[i, j] = np.mean((simulate_vector_host(float(beta_hv), float(beta_vh))[obs_idx, 1] - observed) ** 2)
best_flat = np.argsort(surface, axis=None)[:18]
best_pairs = np.array([(grid[np.unravel_index(idx, surface.shape)[0]], grid[np.unravel_index(idx, surface.shape)[1]]) for idx in best_flat])
best_i, best_j = np.unravel_index(int(np.argmin(surface)), surface.shape)
best_vector = simulate_vector_host(float(grid[best_i]), float(grid[best_j]))


## Pemeriksaan numerik

Pemeriksaan berikut sengaja berada di dalam notebook: eksekusi berhenti bila suatu invarian dasar gagal. Ini bukan bukti bahwa model benar; ini hanya bukti bahwa implementasi memenuhi kontrak numerik terbatasnya.


In [ ]:
np.testing.assert_allclose(truth[:, :3].sum(axis=1), 1.0, atol=2e-10)
np.testing.assert_allclose(truth[:, 3:].sum(axis=1), 1.0, atol=2e-10)
assert np.min(truth) >= 0.0 and np.min(best_simple) >= 0.0
assert np.ptp(best_pairs[:, 0]) > 0.15 and np.ptp(best_pairs[:, 1]) > 0.15
assert np.isfinite(surface).all()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(times[obs_idx], observed, s=18, color="black", label="data manusia sintetis")
axes[0].plot(times, best_simple[:, 1], label="SIR sederhana")
axes[0].plot(times, best_vector[:, 1], label="vektor–inang")
axes[0].set(xlabel="hari", ylabel="prevalensi manusia", title="Dua kelas model")
axes[0].legend(fontsize=8)
image = axes[1].imshow(np.log10(surface + 1e-12), origin="lower", extent=[grid[0], grid[-1], grid[0], grid[-1]], aspect="auto")
axes[1].scatter(best_pairs[:, 1], best_pairs[:, 0], s=10, color="white")
axes[1].set(xlabel="laju vektor→manusia", ylabel="laju manusia→vektor", title="Galat log dan punggung parameter")
fig.colorbar(image, ax=axes[1], label="log10 MSE")
fig.tight_layout()
plt.show()
plt.close(fig)


## Validasi, identifikasi, dan keterbatasan

Keterbatasan awal: Model mengabaikan musim, serotipe, imunitas silang, umur nyamuk, pelaporan kasus, dan struktur ruang; kisi parameter hanya ilustrasi keteridentifikasian praktis.

Jawab sebelum menafsirkan gambar:

1. Besaran apa yang benar-benar dapat diamati, dan bagaimana galat pengukurannya dimodelkan?
2. Parameter mana yang dapat diidentifikasi dari keluaran tersebut? Tunjukkan dengan profil galat, pemisahan latih/uji, atau eksperimen sensitivitas.
3. Invarian atau pola kualitatif apa yang harus tetap benar ketika ukuran langkah, benih acak, atau resolusi diubah?
4. Temukan satu skenario kegagalan model dan jelaskan data tambahan yang diperlukan untuk membedakannya dari model alternatif.


## Daftar periksa reproduksibilitas

- [ ] Gunakan CPython dan versi paket tepat seperti `requirements.lock`.
- [ ] Jalankan ulang dari kernel kosong tanpa jaringan.
- [ ] Pertahankan nilai `SEED` (benih acak), lalu ulangi dengan sedikitnya lima benih acak lain dan laporkan variasinya.
- [ ] Catat setiap perubahan parameter, persamaan, toleransi, serta pembagian data.
- [ ] Pastikan semua uji lulus dan jelaskan mengapa tiap uji relevan.
- [ ] Simpan hasil turunan di luar notebook sumber; notebook distribusi harus tetap tanpa keluaran tersimpan.
- [ ] Bedakan hasil simulasi, data sintetis, dan klaim empiris secara eksplisit.
